In [ ]:
import torch
import torch

from lxt.efficient import monkey_patch_zennit
from basemodel import load_timm_wrapper
from dino_patcher import DINOPatcher

In [ ]:

monkey_patch_zennit(verbose=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


model_wrapper, transforms, data_config = load_timm_wrapper(
        checkpoint_path="/workspaces/vast-gorilla/gorillawatch/models/ViTG-body_face-spac23+24v3-dedup.pth",
        backbone_name="vit_giant_patch14_dinov2.lvd142m",
        embedding_size=256,
        image_size=518,
        finetuned=True,
        device=DEVICE,
        bp_transforms=True,
        model_dtype=torch.float32,
    )

In [ ]:
for name, module in model_wrapper.named_modules():
    key = name
    print(f"{key:50} {module.__class__.__name__}")

In [ ]:
from torchvision.models import vision_transformer

def get_vit_imagenet(device="cuda"):
    """
    Load a pre-trained Vision Transformer (ViT) model with ImageNet weights.

    Parameters:
    device (str): Device to load the model on ('cuda' or 'cpu')

    Returns:
    tuple: (model, weights) - The ViT model and its pre-trained weights
    """
    weights = vision_transformer.ViT_B_16_Weights.IMAGENET1K_V1
    model = vision_transformer.vit_b_16(weights=weights)
    model.eval()
    model.to(device)

    # Deactivate gradients on parameters to save memory
    for param in model.parameters():
        param.requires_grad = False

    return model, weights


# Load the pre-trained ViT model
model, weights = get_vit_imagenet()

In [ ]:
from torch.nn import MultiheadAttention

print("-" * 80)
for name, module in model.named_modules():
    key = name if name else "model" # Handle the top-level module
    print(f"{key:50} {module.__class__.__name__}")

    # Add a special check for MultiheadAttention
    if isinstance(module, MultiheadAttention):
        print(f"{'':52}   Heads: {module.num_heads}, Embed Dim: {module.embed_dim}")
        # Print shape of the combined Q, K, V weight matrix
        print(f"{'':52}   In-Proj Shape: {module.in_proj_weight.shape}")
print("-" * 80)